# Passo 1: Importar as bibliotecas!

In [ ]:
import os
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta
import argparse
import os
import pandas as pd

# Passo 2: Carregar o .env e construir o cliente do youtube

In [ ]:
load_dotenv()

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)  # Para a abordagem 1

# Passo 3.1: Coleta dos vídeos (abordagem 1)
### Coleta vídeos de um canal do YouTube publicados dentro de uma janela de datas específica.
    Utiliza a playlist de 'uploads' do canal para economizar quota da API.
3.1.1: Obter o ID da playlist de uploads do canal


In [ ]:
def coletar_videos(youtube, channel_id, data_inicio, data_fim):
  try:
          # Consulta as informações do canal para obter a lista especial de "uploads"
          res = youtube.channels().list(part="contentDetails", id=channel_id).execute()
          items = res.get("items", [])

          # Tratamento: Caso o ID do canal não exista ou seja inválido
          if not items:
              print(f"[ERRO] Canal {channel_id} não encontrado.")
              return []

          # O YouTube guarda TODOS os vídeos postados por um canal em uma playlist interna de 'uploads'
          uploads_playlist = items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

      except HttpError as e:
          # Trata erros de requisição HTTP (ex: chave de API inválida, sem permissão ou quota estourada)
          print(f"[ERRO] Falha ao buscar canal {channel_id}: {e}")
          return []

3.1.2: Converter os parâmetros de texto (strings) para objetos 'date' do Python

In [ ]:
      # Permite comparar as datas de publicação dos vídeos usando operadores comuns (>, <, ==)
      dt_inicio = datetime.strptime(data_inicio, "%Y-%m-%d").date()
      dt_fim = datetime.strptime(data_fim, "%Y-%m-%d").date()

      videos_coletados = []  # Lista final que armazenará os dicionários com os dados de cada vídeo
      token = None           # Controla a paginação (guardará o ID da próxima página de resultados)

3.1.3: Loop de Paginação (Varredura do mais recente para o mais antigo)

In [ ]:
      while True:
          try:
              # Busca itens da playlist de uploads em lotes de até 50 (máximo permitido por requisição)
              res = youtube.playlistItems().list(
                  part="contentDetails",
                  playlistId=uploads_playlist,
                  maxResults=50,
                  pageToken=token  # Nulo na 1ª chamada; nas demais, recebe a instrução para a próxima página
              ).execute()

              video_ids_da_pagina = []
              atingiu_limite_antigo = False  # Flag de interrupção para interromper o loop prematuramente

              # Iterar sobre cada item (vídeo) retornado na página atual
              for item in res.get("items", []):
                  # Extrai os 10 primeiros caracteres da data no formato ISO ("YYYY-MM-DDTHH:MM:SSZ")
                  raw_date = item["contentDetails"]["videoPublishedAt"]
                  data_publicacao = datetime.strptime(raw_date[:10], "%Y-%m-%d").date()

                  # CASO A: Vídeo mais recente do que a janela desejada -> ignora e continua descendo na lista
                  if data_publicacao > dt_fim:
                      continue

                  # CASO B: Vídeo mais antigo que a data de início -> como a playlist é ordenada por data de postagem,
                  # todos os próximos vídeos serão ainda mais antigos. Então marcamos a flag e interrompemos o loop interno.
                  if data_publicacao < dt_inicio:
                      atingiu_limite_antigo = True
                      break

                  # CASO C: O vídeo está DENTRO da janela de datas -> guarda o ID para coletar os detalhes completos
                  video_ids_da_pagina.append(item["contentDetails"]["videoId"])

3.1.4: Enriquecimento de dados (Coleta de métricas e detalhes em lote)

In [ ]:
              # A playlist só retorna IDs; para pegar visualizações, curtidas, título e tag, fazemos uma nova chamada 'videos().list'
              if video_ids_da_pagina:
                  video_res = youtube.videos().list(
                      part="snippet,statistics,contentDetails",
                      id=",".join(video_ids_da_pagina)  # Envia múltiplos IDs separados por vírgula de uma só vez
                  ).execute()

                  # Processa os detalhes completos retornados para cada vídeo do lote
                  for v in video_res.get("items", []):
                      snippet = v["snippet"]
                      stats = v.get("statistics", {})  # Usa .get() pois vídeos podem ter estatísticas privadas
                      details = v["contentDetails"]

                      # Regra simples para identificar se o vídeo é um YouTube Short baseado na duração em formato ISO 8601
                      # (ex: "PT45S" não tem 'M' de minuto ou 'H' de hora)
                      duracao_iso = details.get("duration", "PT0S")
                      is_short = ("M" not in duracao_iso and "H" not in duracao_iso)

                      # Constrói a estrutura final e adiciona à lista
                      videos_coletados.append({
                          "videoId": v["id"],
                          "channelId": snippet["channelId"],
                          "channelTitle": snippet["channelTitle"],
                          "publishedAt": snippet["publishedAt"],
                          "title": snippet["title"],
                          "description": snippet["description"],
                          "tags": ",".join(snippet.get("tags", [])),  # Transforma a lista de tags em uma string única
                          "categoryId": snippet["categoryId"],
                          "duration": details["duration"],
                          "is_short": is_short,
                          "viewCount": int(stats.get("viewCount", 0)),
                          "likeCount": int(stats.get("likeCount", 0)),
                          "commentCount": int(stats.get("commentCount", 0)),
                          "_entidade_busca": channel_id  # Rastreabilidade do canal pesquisado
                      })

3.1.5: Controle do Loop Principal (Paginação)

In [ ]:

              token = res.get("nextPageToken")  # Obtém o token da próxima página

              # Para o loop principal se:
              # 1. Encontrou um vídeo mais antigo do que a 'data_inicio' (atingiu_limite_antigo = True)
              # 2. Chegou ao fim da playlist e não há próxima página (token é None)
              if atingiu_limite_antigo or not token:
                  break

          except HttpError as e:
              # Caso aconteça um erro ao paginar, registra a falha e encerra a coleta mantendo o que já foi baixado
              print(f"[ERRO] Falha ao paginar vídeos: {e}")
              break

  # Retorna a lista contendo todos os vídeos coletados e estruturados no intervalo selecionado
  return videos_coletados

# Passo 3.2: Coleta dos vídeos (abordagem 2)
### Busca vídeos no YouTube iterando sobre termos de pesquisa (queries) e fatias de tempo.
3.2.1: Estruturas Globais da Função

In [ ]:
# `ids_coletados` (Set): Garante busca O(1) e impede que o mesmo vídeo seja adicionado duas vezes.
ids_coletados = set()
# `video_data` (List): Armazena os dicionários finais que serão exportados para CSV/Dataframe.
video_data = []
# `cota_excedida` (Bool): Flag de controle para interromper requisições caso a API estoure o limite.
cota_excedida = False
try:

3.2.2: Iteração por Palavras-Chave

In [ ]:
for query in queries:
      # Trava de segurança: para o loop de palavras-chave se atingiu a meta de vídeos ou estourou a quota
      if cota_excedida or len(ids_coletados) >= meta_videos:
          break

      print(f"\n--- Iniciando buscas para a palavra-chave: '{query}' ---")
      data_inicio_fatia = data_inicio

3.2.3: Fatiamento Temporal

In [ ]:
      # A API do YouTube limita os resultados a 500 por busca.
      # Fatiar o intervalo em blocos menores (ex: 7 dias) contorna essa limitação do motor de busca.
      while data_inicio_fatia < data_fim:
          if cota_excedida or len(ids_coletados) >= meta_videos:
              break

          # Calcula a data final da fatia atual
          data_fim_fatia = data_inicio_fatia + timedelta(days=intervalo_dias)
          if data_fim_fatia > data_fim:
              data_fim_fatia = data_fim

          # Formata as datas para o padrão ISO 8601 exigido pela API do YouTube (YYYY-MM-DDTHH:MM:SSZ)
          published_after = data_inicio_fatia.strftime('%Y-%m-%dT00:00:00Z')
          published_before = data_fim_fatia.strftime('%Y-%m-%dT23:59:59Z')

          print(f"Buscando de {data_inicio_fatia.strftime('%d/%m/%Y')} até {data_fim_fatia.strftime('%d/%m/%Y')}...")

3.2.4: Montagem da Requisição HTTP

In [ ]:
          request = youtube.search().list(
              q=query,
              part="id,snippet",
              type="video",
              maxResults=50,  # Máximo de itens retornados por página
              publishedAfter=published_after,
              publishedBefore=published_before
          )

3.2.5: Paginação do YouTube (NextPageToken)

In [ ]:
          # `request` torna-se `None` quando não existirem mais páginas para a fatia de tempo atual
          while request is not None:
              # Executa a requisição HTTP no Google Cloud
              response = request.execute()

              # Iteração pelos itens retornados no lote atual
              for item in response.get('items', []):
                  # Validação de Segurança: descarta canais ou playlists que possam vir no payload
                  if 'videoId' not in item.get('id', {}):
                      continue

                  vid = item['id']['videoId']

                  # Deduplicação: processa apenas vídeos inéditos na sessão
                  if vid not in ids_coletados:
                      ids_coletados.add(vid)
                      snippet = item.get('snippet', {})

                      # Mapeamento do dicionário final de dados
                      video_data.append({
                          'videoId': vid,
                          'termo_busca': query,
                          'title': snippet.get('title'),
                          'channelTitle': snippet.get('channelTitle'),
                          'channelId': snippet.get('channelId'),
                          'publishedAt': snippet.get('publishedAt'),
                          'short_description': snippet.get('description')
                      })

              print(f" > {len(ids_coletados)} IDs únicos coletados no total.")

              # Trava interna: se atingiu a meta dentro do loop de paginação, quebra o processamento
              if len(ids_coletados) >= meta_videos:
                  break

              # Prepara a requisição da próxima página (avança o token automaticamente)
              request = youtube.search().list_next(request, response)

          # Avança a janela temporal para a próxima fatia de dias
          data_inicio_fatia = data_fim_fatia + timedelta(days=1)

3.2.6: Tratamento de Erros

In [ ]:
except HttpError as e:
        # Erro 403: Geralmente indica estouro de cota (10.000 unidades/dia) ou chave inválida
        if e.resp.status in [403]:
            print("\n[AVISO] Cota de API excedida ou chave inválida! Interrompendo a busca...")
        else:
            print(f"\nErro HTTP: {e}")
    except Exception as e:
        # Captura erros imprevistos (ex: queda de conexão, interrupções do sistema)
        print(f"\nErro inesperado: {e}")

    # Retorna os dados parciais ou completos acumulados com segurança até o ponto da parada
    return video_data

# Passo 4: Coleta dos comentários
### Extrai comentários de nível principal (top-level) de um vídeo específico do YouTube.
    
    Parâmetros:
        youtube: Instância do cliente da API do YouTube v3.
        video_id (str): O ID exclusivo do vídeo do YouTube.
        channel_title (str): O nome do canal para fins de rastreabilidade/metadados.
        limite_paginas (int): Limite máximo de páginas a consultar (evita consumo excessivo de quota).
        
    Retorna:
        list: Lista de dicionários contendo os comentários extraídos e suas métricas.

4.1: Inicialização das Estruturas de Controle

In [ ]:
comentarios = []  # Lista acumuladora onde serão salvos os dicionários de comentários
token = None      # Token da API para controlar a paginação (começa como None na 1ª requisição)
paginas = 0       # Contador de páginas processadas

4.2: Loop de Paginação com Limite de Segurança

In [ ]:
# O loop executa até varrer todos os comentários (token ser None) ou atingir o limite de páginas estipulado
while paginas < limite_paginas:
    try:

4.3: Construção da Requisição HTTP

In [ ]:
        # O endpoint 'commentThreads().list' recupera as "threads" principais de comentários
        req = youtube.commentThreads().list(
            part="snippet",             # Traz os detalhes do comentário (texto, autor, curtidas, etc.)
            videoId=video_id,           # ID do vídeo alvo
            maxResults=100,             # Máximo de itens permitidos por página pela API (limite do YouTube)
            pageToken=token,            # Aponta para a página atual do lote de comentários
            textFormat="plainText"      # Remove tags HTML do texto (útil para análise de sentimento/NLP)
        )
        res = req.execute()  # Envia a requisição e guarda a resposta JSON em 'res'

4.4: Iteração e Extração do Payload

In [ ]:
        for item in res.get("items", []):
                # Navega na estrutura aninhada da resposta até o nó do comentário principal (Top Level)
                top = item["snippet"]["topLevelComment"]["snippet"]

                # Montagem da estrutura de dados simplificada do comentário
                comentarios.append({
                    "commentId": item["id"],                              # ID único do comentário no YouTube
                    "videoId": video_id,                                  # ID do vídeo associado (Chave Estrangeira)
                    "channelTitle": channel_title,                        # Nome do canal criador do vídeo
                    "text": top["textOriginal"],                          # Conteúdo em texto puro do comentário
                    "likeCount": int(top.get("likeCount", 0)),             # Total de curtidas recebidas no comentário
                    "replyCount": int(item["snippet"].get("totalReplyCount", 0)), # Quantidade de respostas nessa thread
                    "publishedAt": top["publishedAt"]                     # Data e hora da postagem (ISO 8601)
                })

4.5: Atualização dos Contadores e Avanço da Paginação

In [ ]:
        token = res.get("nextPageToken")  # Obtém a chave para a próxima página de resultados
        paginas += 1                      # Incrementa a contagem de páginas processadas

        # Se a chave 'nextPageToken' não existir no JSON, significa que não há mais comentários para ler
        if not token:
            break

4.6: Tratamento de Erros de API (HttpError)

In [ ]:
    except HttpError as e:
        # O código HTTP 403 neste endpoint geralmente indica que o criador DESATIVOU os comentários no vídeo
        if e.resp.status == 403:
            print(f"[AVISO] Comentários desativados para o vídeo {video_id}")
        else:
            # Trata outros erros HTTP (como erro 404 de vídeo removido ou estouro de cota)
            print(f"[ERRO] Falha ao buscar comentários do vídeo {video_id}: {e}")
        break  # Interrompe a execução para este vídeo e evita tentar as próximas páginas

4.7: Captura de Erros Genéricos

In [ ]:
    except Exception as e:
        # Captura falhas inesperadas (ex: falha de rede/timeout ou alteração na chave dos dados)
        print(f"[ERRO] Falha inesperada ao buscar comentários: {e}")
        break

# Retorna os comentários capturados com segurança até a interrupção ou conclusão da leitura
return comentarios

# Passo 5: Main!

Como pegar o channel_id do canal desejado:

    1: Vá até a página principal do canal desejado
    2: Aperte "Ctrl + U"
    3: Aperte "Ctrl + F"
    4: Busque por "browse_id" ou "channel_id"

In [ ]:
# Definindo período e parâmetros de busca
dt_inicio = datetime(2011, 12, 14)
dt_fim = datetime(2026, 9, 2)
str_inicio = dt_inicio.strftime("%Y-%m-%d")
str_fim = dt_fim.strftime("%Y-%m-%d")



# A) Coleta vídeos por Canal
id_canal = "UCcabW7890RKJzL968QWEykA" #channel_id do cs50! recomendação pessoal! :D
print(f"\n[1/3] Coletando vídeos do canal ID: {id_canal}...")
videos_canal = coletar_videos(youtube, channel_id=id_canal, data_inicio=str_inicio, data_fim=str_fim)



# B) Coleta vídeos por Palavra-Chave (Queries)
termos_busca = ["web scraping", "pandas tutorial", "cs50"]
print(f"\n[2/3] Buscando vídeos por palavras-chave: {termos_busca}...")
videos_busca = extrair_videos_por_buscas(
    youtube=youtube,
    queries=termos_busca,
    data_inicio=dt_inicio,
    data_fim=dt_fim,
    meta_videos=20,
    intervalo_dias=30
)


5.1 Manipulação dos vídeos com pandas

In [ ]:
print("\n[3/3] Unificando e tratando dados de vídeos com Pandas...")

# Transforma as listas de dicionários em DataFrames do Pandas
df_canal = pd.DataFrame(videos_canal)
df_busca = pd.DataFrame(videos_busca)

# Concatena (combina) os dois DataFrames em um só
df_videos = pd.concat([df_canal, df_busca], ignore_index=True)

if df_videos.empty:
    print("Nenhum vídeo foi coletado. Encerrando execução.")
    return

# A) Deduplicação: Remove vídeos duplicados caso tenham vindo por ambas as buscas
df_videos.drop_duplicates(subset=["videoId"], inplace=True)

# B) Limpeza e Tratamento de Tipos
# Garante que colunas de métricas sejam numéricas (preenchendo valores nulos com 0)
colunas_numericas = ["viewCount", "likeCount", "commentCount"]
for col in colunas_numericas:
    if col in df_videos.columns:
        df_videos[col] = pd.to_numeric(df_videos[col], errors="coerce").fillna(0).astype(int)

# C) Criação de Colunas Calculadas
# Exemplo: Calcula a taxa de engajamento (likes por visualização)
if "likeCount" in df_videos.columns and "viewCount" in df_videos.columns:
    df_videos["taxa_engajamento"] = (df_videos["likeCount"] / df_videos["viewCount"].replace(0, 1) * 100).round(2)

5.2 Coleta dos comentários

In [ ]:
todos_comentarios = []
print("\nExtraindo comentários para os vídeos coletados...")

# Iteramos pelas linhas do DataFrame de vídeos usando .itertuples()
for row in df_videos.itertuples():
    # Só tenta buscar comentários se o vídeo tiver ao menos 1 comentário cadastrado
    comment_count = getattr(row, "commentCount", 0)
    if comment_count > 0:
        print(f" -> Coletando comentários do vídeo: {row.videoId}")
        coments = coletar_comentarios(
            youtube=youtube,
            video_id=row.videoId,
            channel_title=getattr(row, "channelTitle", "Desconhecido"),
            limite_paginas=2  # Limite pequeno para testes no laboratório
        )
        todos_comentarios.extend(coments)

# Converte os comentários para DataFrame
df_comentarios = pd.DataFrame(todos_comentarios)

5.3 Manipulação dos comentários com pandas

In [ ]:
if not df_comentarios.empty:
    # A) Limpeza de caracteres especiais e quebras de linha nos textos
    df_comentarios["text"] = df_comentarios["text"].str.replace(r"[\r\n]+", " ", regex=True).str.strip()

    # B) Ordenação: Mostra primeiro os comentários mais curtidos
    df_comentarios.sort_values(by="likeCount", ascending=False, inplace=True)

5.4 Exportando os dados para um .csv

In [ ]:
print("\nSalvando arquivos CSV...")

# exporta o DataFrame de Vídeos
df_videos.to_csv(
    "youtube_videos_master.csv",
    index=False,           # Remove a coluna de índice numérico do Pandas (0, 1, 2...)
    encoding="utf-8-sig"
)
print(f"'youtube_videos_master.csv' gerado com {len(df_videos)} registros.")

# exporta o DataFrame de Comentários
if not df_comentarios.empty:
    df_comentarios.to_csv(
        "youtube_comments_master.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print(f"'youtube_comments_master.csv' gerado com {len(df_comentarios)} registros.")

# **Fim! :D**
## Desafio 1: Criar função para atualizar diariamente os comentários
    Dicas:
    1: Ler o CSV existente.
    2: Extrair os IDs dos vídeos existentes.
    3: Buscar apenas as novas atualizações.
    4: Salvar o CSV atualizado.
## Desafio 2: Criar função para atualizar diariamente os vídeos

## Desafio 3: Automatizar a atualização diária de videos e comentários

## Desafio 4: Elaborar uma forma de obter as legendas/transcrições do youtube